In [ ]:
import os
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

In [2]:
HF_TOKEN = os.environ["HF_TOKEN"]

## Extract the drone dataset

`data/archive.zip` holds 357 MicaSense-style captures (single-band `GRE` / `RED` / `REG` / `NIR` `.tif` files) plus 182 true-color `.JPG` composites. Extract once into `data/`.

In [ ]:
DATA_DIR = Path("../data") if Path("../data/archive.zip").exists() else Path("data")
RGB_DIR = DATA_DIR / "rgb-images"

if not RGB_DIR.exists():
    with zipfile.ZipFile(DATA_DIR / "archive.zip") as zf:
        zf.extractall(DATA_DIR)

rgb_paths = sorted(RGB_DIR.glob("*.JPG")) or sorted(RGB_DIR.glob("*.jpg"))
print(f"{len(rgb_paths)} RGB captures available")

## Load SegFormer-b0

Pretrained on ADE20K (150 general scene classes — not agriculture-specific). This confirms the pipeline runs end-to-end on your imagery before any crop/weed labels exist.

In [ ]:
MODEL_ID = "nvidia/segformer-b0-finetuned-ade-512-512"

processor = SegformerImageProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = SegformerForSemanticSegmentation.from_pretrained(MODEL_ID, token=HF_TOKEN)
model.eval()

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model.to(device)
print(f"Loaded {MODEL_ID} on {device}")

## Run inference on a sample capture

In [ ]:
sample_path = rgb_paths[0]
image = Image.open(sample_path).convert("RGB")

inputs = processor(images=image, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits  # (1, num_labels, H/4, W/4)
upsampled = torch.nn.functional.interpolate(
    logits, size=image.size[::-1], mode="bilinear", align_corners=False
)
segmentation = upsampled.argmax(dim=1)[0].cpu().numpy()

print(f"{sample_path.name} -> segmentation map {segmentation.shape}, {len(np.unique(segmentation))} classes detected")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(image)
axes[0].set_title(sample_path.name)
axes[0].axis("off")

axes[1].imshow(image)
axes[1].imshow(segmentation, alpha=0.55, cmap="tab20")
axes[1].set_title("SegFormer-b0 (ADE20K classes)")
axes[1].axis("off")

fig.tight_layout()
plt.show()

## Fine-tuning on your own classes

ADE20K classes (wall, sky, tree, grass, ...) aren't crop/weed/soil labels — useful to confirm the pipeline runs on your imagery, not to draw agronomic conclusions yet. To specialize the model:

1. Label a subset of tiles with your target classes (e.g. `soil`, `crop`, `weed`).
2. Swap the classifier head to your class count.
3. Fine-tune on your labeled tiles.

This dataset doesn't have masks yet — the cell below is the head-swap + training-loop skeleton, ready to wire up once labels exist.

In [ ]:
# --- Fine-tuning skeleton (needs labeled masks — none exist in this dataset yet) ---

CLASS_NAMES = ["background", "crop", "weed"]  # placeholder — replace with your taxonomy

id2label = dict(enumerate(CLASS_NAMES))
label2id = {v: k for k, v in id2label.items()}

finetune_model = SegformerForSemanticSegmentation.from_pretrained(
    MODEL_ID,
    num_labels=len(CLASS_NAMES),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
    token=HF_TOKEN,
)

# class SegmentationDataset(torch.utils.data.Dataset):
#     """Pairs a `rgb-images/*.JPG` (or a band composite) with a {background, crop, weed} mask."""
#     def __init__(self, image_paths, mask_paths, processor):
#         self.image_paths = image_paths
#         self.mask_paths = mask_paths
#         self.processor = processor
#
#     def __len__(self):
#         return len(self.image_paths)
#
#     def __getitem__(self, idx):
#         image = Image.open(self.image_paths[idx]).convert("RGB")
#         mask = Image.open(self.mask_paths[idx])
#         return self.processor(images=image, segmentation_maps=mask, return_tensors="pt")
#
# Once labeled tiles exist, wrap them in the Dataset above and fine-tune `finetune_model`
# with a standard PyTorch loop or `transformers.Trainer`.

print(f"Head swapped to {len(CLASS_NAMES)} classes — ready to fine-tune once masks exist")